# Chapter 14 — 지도학습 기반 공격 탐지 (실습)

## 실습 목표
1. NSL-KDD 합성 데이터 + 5개 범주 라벨
2. 전처리 Pipeline
3. LogReg / SVM / RF 학습
4. ROC-AUC, Confusion Matrix
5. pickle 저장 + 새 데이터 예측

In [ ]:
# 파트별 노트북은 같은 종류의 공통 데이터·모델 작업 폴더를 사용합니다.
from pathlib import Path as _BookPath
import os as _book_os
_book_cwd = _BookPath.cwd()
if (_book_cwd.name in {"part1", "part2", "part3", "part4"}
        and _book_cwd.parent.name in {"instructor", "exercises", "solutions"}):
    _book_os.chdir(_book_cwd.parent)


In [ ]:
# === 한글 폰트 설정 (matplotlib) ===
from pathlib import Path
import matplotlib
from matplotlib import font_manager
for fp in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf',
           '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf']:
    if Path(fp).exists():
        font_manager.fontManager.addfont(fp)
        matplotlib.rcParams['font.family'] = font_manager.FontProperties(fname=fp).get_name()
        break
matplotlib.rcParams['axes.unicode_minus'] = False

In [ ]:
# 첫 사용시에만 주석 해제하여 실행 (Colab/venv 환경에 따라 일부는 이미 설치됨)
# !pip install -q pandas
# !pip install -q numpy
# !pip install -q matplotlib
# !pip install -q seaborn
# !pip install -q scikit-learn

In [ ]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, classification_report, confusion_matrix,
                             roc_curve)
rng = np.random.default_rng(42)

---
## Step 1. 데이터 준비 — NSL-KDD 합성

In [ ]:
# ── 실습 과제 ──
# 목표: Step 1. 데이터 준비 — NSL-KDD 합성
# 만들 결과: make_rows(), df_train, df_test, df_train['label_bin'], df_test['label_bin'], atk
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
def make_rows(n, label, dur, src_b, srv, serr, proto, svc):
    return ...
        'duration':     rng.exponential(scale = ...
        ...
        'dst_bytes':    rng.integers(0, 100, size = ...
        'count':        rng.integers(1, 200, size = ...
        ...
        'rerror_rate':  rng.uniform(0, 0.1, size = ...
        'same_srv_rate':rng.uniform(0.8, 1.0, size = ...
        'diff_srv_rate':rng.uniform(0, 0.1, size = ...
        'protocol_type':rng.choice(proto, size = ...
        'service':      rng.choice(svc,   size = ...
        'flag':         rng.choice(['SF','S0','REJ'], size = ...
        ...

df_train = ...
    ...
], ignore_index = ...
df_test = ...
    ...
], ignore_index = ...

df_train['label_bin'] = ...
df_test['label_bin'] = ...

# --- 현실화: 실제 트래픽은 정상/공격 경계가 겹친다 (합성 데이터가 100% 분리되어 AUC=1.0 이 나오는 것을 방지) ---
for _df in (df_train, df_test):
    atk = ...
    b1 = ...
    _df.loc[b1, 'dst_bytes'] = ...
    _df.loc[b1, 'flag'] = ...
    _df.loc[b1, 'serror_rate'] = ...
    _df.loc[b1, 'same_srv_rate'] = ...
    nrm = ...
    b2 = ...
    _df.loc[b2, 'serror_rate'] = ...
    _df.loc[b2, 'dst_bytes'] = ...
    flip = ...
    _df.loc[flip, 'label_bin'] = ...

print(f'train = ...
df_train['label_5class'].value_counts(normalize = ...
```
</details>


---
## Step 2. 전처리 Pipeline

In [ ]:
# ── 실습 과제 ──
# 목표: Step 2. 전처리 Pipeline
# 만들 결과: NUM, CAT, preprocessor, X_train, X_test
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
NUM = ...
       ...
CAT = ...

preprocessor = ...
    ...
    ('cat', OneHotEncoder(handle_unknown = ...
...
X_train = ...
X_test = ...
y_train, y_test = ...
print(f'X_train.shape = ...
```
</details>


---
## Step 3. 모델 3종 학습

In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 모델 3종 학습
# 만들 결과: model1, model2, model3, random_state
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
model1 = ...
model2 = ...
model3 = ...
                                random_state = ...
for name, m in [('LogReg', model1), ('SVM', model2), ('RF', model3)]:
    print(f'{name}: train = ...
```
</details>


In [ ]:
# ── 실습 과제 ──
# 목표: Step 3. 모델 3종 학습
# 만들 결과: feat_names, imp
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
# 설명가능성 — RandomForest 가 어떤 피처로 공격을 판단하는가
import matplotlib.pyplot as plt
feat_names = ...
imp = ...
fig, ax = ...
...
```
</details>


---
## 앙상블·부스팅 — 모델 #4~#5 + Voting/Stacking

단일 모델(LogReg·SVM·RF) 위에 **부스팅(HistGradientBoosting·XGBoost)** 과 **Voting·Stacking** 앙상블을 더해 성능을 비교한다.
- Bagging(분산↓, RF) vs Boosting(편향↓, XGBoost) vs Voting/Stacking(이질 결합)
- 아래 셀에서 model4(HistGB)·model5(XGBoost)·voting·stacking 을 학습하고 test AUC 를 출력한다.


In [ ]:
# ── 실습 과제 ──
# 목표: 앙상블·부스팅 — 모델 #4~#5 + Voting/Stacking — 단일 모델(LogReg·SVM·RF) 위에 **부스팅(HistGradientBoosting·XGBoost)** 과 **Voting·Stacking** 앙상블을 더해 성능을 비교한다.
# 만들 결과: model4, max_iter, model5, n_estimators, colsample_bytree, eval_metric
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.


<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
from sklearn.ensemble import HistGradientBoostingClassifier, VotingClassifier, StackingClassifier

# 모델 #4 — Gradient Boosting (sklearn, 설치 불필요)
model4 = ...
    max_iter = ...

# 모델 #5 — XGBoost (미설치 환경이면 HistGB 로 대체)
try:
    from xgboost import XGBClassifier
    neg, pos = ...
    model5 = ...
        n_estimators = ...
        colsample_bytree = ...
        eval_metric = ...
except ImportError:
    print('xgboost 미설치 — model5 를 HistGB 로 대체'); model5 = ...

# 이질 모델 결합
voting = ...
    estimators = ...
    voting = ...
stacking = ...
    estimators = ...
    final_estimator = ...

for name, m in [('HistGB', model4), ('XGBoost', model5), ('Voting', voting), ('Stacking', stacking)]:
    auc = ...
    print(f'{name:9s}: test AUC = ...
```
</details>


---
## 교차검증과 하이퍼파라미터 튜닝
단일 분할 점수는 분할 운에 좌우된다. StratifiedKFold 교차검증으로 안정 추정하고 GridSearchCV 로 최적 파라미터를 찾는다. SMOTE 는 소수 클래스 오버샘플링(설치 시).


In [ ]:
# ── 실습 과제 ──
# 목표: 교차검증과 하이퍼파라미터 튜닝 — 단일 분할 점수는 분할 운에 좌우된다. StratifiedKFold 교차검증으로 안정 추정하고 GridSearchCV 로 최적 파라미터를 찾는다. SMOTE 는 소수 클래스 오버샘플링(설치 시).
# 만들 결과: cv_pipe, skf, cv, param_grid, gs, pipe
# 산출: 결과 값 출력
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.


<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV

from sklearn.pipeline import Pipeline
from sklearn.base import clone
cv_pipe = ...
skf = ...
cv = ...
...

param_grid = ...
gs = ...
                  param_grid, cv = ...
...

# SMOTE 오버샘플링 (imbalanced-learn 설치 시)
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    pipe = ...
                        ('clf', RandomForestClassifier(random_state = ...
    f1 = ...
    ...
except ImportError:
    ...
```
</details>


---
## Step 4. ROC-AUC + Confusion Matrix

In [ ]:
# ── 실습 과제 ──
# 목표: Step 4. ROC-AUC + Confusion Matrix
# 만들 결과: proba, cm, xticklabels
# 산출: 그래프 시각화
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
import seaborn as sns
fig, axes = ...
...
for name, m, c in [('LogReg', model1, '#1a5f7a'), ('SVM', model2, '#0984e3'), ('RF', model3, '#e74c3c')]:
    proba = ...
    fpr, tpr, _ = ...
    axes[0].plot(fpr, tpr, label = ...
...

cm = ...
sns.heatmap(cm, annot = ...
            xticklabels = ...
...

tn, fp, fn, tp = ...
print(f'Precision = ...
```
</details>


---
## Step 5. pickle 저장 + 새 데이터 예측

In [ ]:
# ── 실습 과제 ──
# 목표: Step 5. pickle 저장 + 새 데이터 예측
# 만들 결과: saved, new, X_new, new['attack_prob']
# 아래에 직접 구현하세요. 막히면 다음 셀의 '힌트 보기'(구조 스켈레톤)를 참고합니다.

# 20장에서 같은 보류 표본으로 연결을 확인한다. 운영 성능 평가용 자료는 아니다.
import json
holdout = df_test[NUM + CAT].copy()
prob = saved['model'].predict_proba(saved['preprocessor'].transform(holdout))[:, 1]
rows = []
for route, mask in [('pass', prob < 0.3), ('llm_agent', (prob >= 0.3) & (prob <= 0.9)),
                    ('playbook', prob > 0.9)]:
    indices = np.flatnonzero(mask)
    if len(indices):
        row = json.loads(holdout.iloc[int(indices[0])].to_json())
        rows.append({'expected_route': route, 'features': row})
Path('models/nsl_kdd_holdout.json').write_text(json.dumps(rows, indent=2), encoding='utf-8')
print('20장 연결 확인용 보류 표본:', [r['expected_route'] for r in rows])


<details><summary>💡 <b>힌트 보기</b> (막혔을 때만)</summary>

> 정답이 아닌 **구조 스켈레톤**입니다. 변수명·제어흐름·주석만 노출되고 실제 값/로직은 `...` 로 가려둠.

```python
...
with open(str(Path.cwd() / 'models' / 'nsl_kdd_rf.pickle'), 'wb') as f:
    ...

with open(str(Path.cwd() / 'models' / 'nsl_kdd_rf.pickle'), 'rb') as f:
    saved = ...

# 새 5건 시뮬레이션
new = ...
X_new = ...
new = ...
new['attack_prob'] = ...
...
```
</details>


## 보완 실습. 학습곡선

14.6.4절과 같은 코드입니다. 전처리는 폴드별로 학습하며 테스트셋은 사용하지 않습니다. 표본 수·트리 깊이에 따른 학습/검증 차이를 비교합니다.

**확인 과제**

1. `max_depth=3`과 기본 설정을 같은 폴드로 비교합니다.
2. 학습·검증 F1의 차이를 표로 기록하고 표본 증가의 효과가 일관적인지 확인합니다.
3. 이 과정에서 최종 테스트셋을 사용하지 않는 이유를 설명합니다.

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.model_selection import learning_curve, StratifiedKFold

curve_model = Pipeline([
    ('preprocess', clone(preprocessor)),
    ('model', RandomForestClassifier(n_estimators=50, random_state=42,
                                     class_weight='balanced', n_jobs=1)),
])
sizes, train_scores, valid_scores = learning_curve(
    curve_model, df_train[NUM + CAT], df_train['label_bin'],
    train_sizes=[0.2, 0.5, 1.0],
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=42),
    scoring='f1', shuffle=True, random_state=42, n_jobs=1,
)
curve = pd.DataFrame({'학습 표본 수': sizes,
                      '학습 F1': train_scores.mean(axis=1),
                      '검증 F1': valid_scores.mean(axis=1)})
print(curve.round(3).to_string(index=False))


---
## 정리
- KISIA Part 5 의 5단계 프레임이 보안 데이터에 그대로 적용된다.
- RF 가 일반적으로 가장 좋은 기준 모델.
- pickle 로 전처리기·모델을 함께 저장하면 운영 시 일관성 보장.